# MeS transition states, IRC selection, and solvent single points

Run each section only after its Gaussian jobs finish. IRC selection retains the original criteria and file-moving behavior (IRC_SELECT_DRY_RUN controls preview versus move). External calculation outputs are not generated by running any other notebook implicitly.

Original notebooks and all saved outputs are preserved in `notebooks/archive/discovery_before_split_20260928/`. This reorganized notebook has not been executed.


In [ ]:
from pathlib import Path
import os
import sys
from IPython.display import display

start = Path.cwd().resolve()
REPO_ROOT = next((p for p in (start, *start.parents)
                  if (p / "Code" / "model_train.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Start Jupyter within the CycloAddGenerator repository.")
for folder in (REPO_ROOT, REPO_ROOT / "notebooks",
               REPO_ROOT / "notebooks/04_discovery", REPO_ROOT / "notebooks/05_mechanism"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
os.chdir(REPO_ROOT)


In [ ]:
from _mes_common import *
from _workflow_paths import require_file
display(pd.DataFrame({"path": [str(INPUT_CSV), str(OUTPUT_DIR), str(ALL_SMILES_CSV)],
                      "exists": [INPUT_CSV.is_file(), OUTPUT_DIR.is_dir(), ALL_SMILES_CSV.is_file()]}))


## C-S scans / TS-search inputs

In [ ]:
RUN_TS_SCAN = True
RUN_ALL = True
SCAN_MODE = "ts"  # "ts" or "constrained"

if RUN_TS_SCAN:
    scan_manifest, scan_failed = generate_methanethiolate_ts_scan_inputs(
        product_log_dir=OUTPUT_DIR / "product",
        product_mol_dir=OUTPUT_DIR / "mol",
        scan_dir=OUTPUT_DIR / "ts_scan",
        
        factor_start=1.00,
        factor_stop=1.50,
        factor_step=0.1,
        mode=SCAN_MODE,
        limit=None if RUN_ALL else LIMIT,
        overwrite=False,
    )
    display(scan_manifest.head())


In [ ]:
main.error_improve(str(OUTPUT_DIR), "ts_scan")


## IRC inputs

In [ ]:
# Generate IRC inputs from all completed TS logs in the flat ts_scan folder.
RUN_IRC = True
TS_SCAN_NAME = "ts_scan"
IRC_OUTPUT_NAME = "irc_scan"
IRC_METHODS = None  # Use Code.cycle_process.ts_to_irc defaults when None.

if RUN_IRC:
    ts_log_dir = OUTPUT_DIR / TS_SCAN_NAME
    ts_logs = sorted(ts_log_dir.glob("*.log"))
    print(f"Found {len(ts_logs)} TS log files in {ts_log_dir}")
    if not ts_logs:
        raise FileNotFoundError(f"No TS .log files found in {ts_log_dir}")

    main.reaction_calc_irc(
        str(OUTPUT_DIR),
        ts_name=TS_SCAN_NAME,
        irc_name=IRC_OUTPUT_NAME,
        methods=IRC_METHODS,
        calc_all=True,
    )
    print(f"IRC inputs were written to {OUTPUT_DIR / IRC_OUTPUT_NAME}")


## Select successful IRC pairs

In [ ]:
RUN_IRC_SUCCESS_SELECT = True
IRC_SELECT_DRY_RUN = False  # Set False after checking the manifest to move files.

if RUN_IRC_SUCCESS_SELECT:
    success_manifest, success_failed, moved_files = select_successful_methanethiolate_irc(
        output_dir=OUTPUT_DIR,
        ts_name="ts_scan",
        irc_name="irc_scan",
        success_name="ts_success",
        lower_factor=0.75,
        upper_factor=1.25,
        overwrite=False,
        dry_run=IRC_SELECT_DRY_RUN,
    )
    display(success_manifest.head())
    display(success_manifest[success_manifest["success"]].head())


## Product and successful-TS solvent single points

In [ ]:
# format_change.block_to_gjf writes the leading "#p"; keep the method string without "#p".
SPE_SOLVENT_METHOD = "b3lyp/6-311+g(d,p) em=gd3bj scrf=(smd, solvent=water)"
RUN_PRODUCT_SPE = True
RUN_SUCCESS_TS_SPE = True

if RUN_PRODUCT_SPE:
    main.SPE_DFT_calc(
        str(OUTPUT_DIR),
        opt_name="product",
        eng_name="product_eng",
        method=SPE_SOLVENT_METHOD,
    )

if RUN_SUCCESS_TS_SPE:
    ts_success_spe_manifest, ts_success_spe_failed = write_success_ts_spe_inputs(
        output_dir=OUTPUT_DIR,
        success_name="ts_success",
        eng_name="ts_success_eng",
        method=SPE_SOLVENT_METHOD,
        overwrite=False,
    )
    display(ts_success_spe_manifest.head())


## Gaussian follow-up

After TS calculations finish:

1. Retain structures with exactly one imaginary frequency.
2. Inspect that normal mode to ensure it is dominated by formation/cleavage of
   the selected `4pi-C--S` bond.
3. Run IRC in both directions, for example with the existing project helper:

```python
main.reaction_calc_irc(
    str(OUTPUT_DIR),
    ts_name="ts_scan",
    irc_name="irc_scan",
    calc_all=True,
)
```

Successful TS/IRC pairs can be screened with `select_successful_methanethiolate_irc`.
It writes `ts_success/ts_success_manifest.csv`; set `IRC_SELECT_DRY_RUN = False`
 in the run cell to move files into `ts_success`.

Solvent single-point inputs are written by the SPE cell into `product_eng` and
`ts_success_eng` using `b3lyp/6-311+g(d,p) em=gd3bj scrf=(smd, solvent=water)`.

The reactant reference input is `MeS_reference.gjf`; use the same electronic
structure level as the diene and TS when calculating activation free energies.
